In [2]:
import requests
import pandas as pd
from pathlib import Path

## Project Set up

In [3]:
API_URL = (
    "https://api.waterdata.usgs.gov/ogcapi/v1/"
    "collections/daily/items"
)

START_DATE = "2025-01-01"
END_DATE = "2025-12-31"

SITE = "USGS-01435000"

# Urban/rural classification still needs verification
AREA_TYPE = "Unknown"

# Verified USGS time series from the metadata results
TIME_SERIES = {
    "streamflow": "ea0befb1e5684ae1a1efa98337f514ca",
    "water_temperature": "f80fcea1c1f14bd095aca16f1c6f0d47"
}

PARAMETERS = {
    "streamflow": "00060",
    "water_temperature": "00010"
}

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print("Project: Streamflow and Water Quality")
print("Monitoring station:", SITE)


Project: Streamflow and Water Quality
Monitoring station: USGS-01435000


## PART 2: COLLECT DATA FROM USGS

In [4]:
def fetch_usgs_data(site, parameter, start_date, end_date):
    """Retrieve daily mean observations from the USGS API."""

    params = {
        "monitoring_location_id": site,
        "parameter_code": parameter,
        "statistic_id": "00003",
        "datetime": f"{start_date}/{end_date}",
        "limit": 500,
        "f": "json"
    }

    records = []
    url = API_URL

    with requests.Session() as session:
        while url:
            response = session.get(
                url,
                params=params,
                timeout=40
            )

            response.raise_for_status()
            result = response.json()

            for feature in result.get("features", []):
                records.append(feature["properties"])

            next_url = None

            for link in result.get("links", []):
                if link.get("rel") == "next":
                    next_url = link["href"]
                    break

            url = next_url
            params = None

    return pd.DataFrame(records)


# Retrieve both variables
all_data = []

for variable, parameter in PARAMETERS.items():

    data = fetch_usgs_data(
        SITE,
        parameter,
        START_DATE,
        END_DATE
    )

    print(f"\n{variable}: {len(data)} records retrieved")

    if data.empty:
        raise ValueError(f"No data found for {variable}")

    # Use only the verified time series
    selected_id = TIME_SERIES[variable]

    data = data[
        data["time_series_id"] == selected_id
    ].copy()

    print(f"{variable}: {len(data)} records after filtering")

    if data.empty:
        raise ValueError(
            f"No records matched selected series for {variable}"
        )

    data["variable"] = variable
    data["site"] = SITE
    data["area_type"] = AREA_TYPE

    all_data.append(data)

raw_data = pd.concat(all_data, ignore_index=True)

print("\nTotal selected observations:", len(raw_data))


streamflow: 365 records retrieved
streamflow: 365 records after filtering

water_temperature: 2129 records retrieved
water_temperature: 335 records after filtering

Total selected observations: 700


## PART 3: CLEAN THE DATA

In [5]:
clean_data = raw_data.copy()

# Convert dates to datetime
clean_data["date"] = pd.to_datetime(
    clean_data["time"],
    errors="coerce"
).dt.normalize()

# Convert measurements to numbers
clean_data["value"] = pd.to_numeric(
    clean_data["value"],
    errors="coerce"
)

# Remove missing values
clean_data = clean_data.dropna(
    subset=["date", "value"]
)

# Check measurement units
print("\nMeasurement units:")
print(
    clean_data.groupby("variable")[
        "unit_of_measure"
    ].unique()
)

# Check for duplicate observations
duplicate_count = clean_data.duplicated(
    subset=["site", "date", "variable"]
).sum()

print("\nDuplicate observations:", duplicate_count)

# Prevent conflicting measurements from being merged
if duplicate_count > 0:
    raise ValueError(
        "Duplicate dates remain within a selected time series."
    )

print("Clean observations:", len(clean_data))


Measurement units:
variable
streamflow           [ft^3/s]
water_temperature      [degC]
Name: unit_of_measure, dtype: object

Duplicate observations: 0
Clean observations: 700


## PART 4: MERGE STREAMFLOW AND TEMPERATURE

In [9]:
# Convert long-format observations to wide format
merged_data = clean_data.pivot(
    index=["site", "area_type", "date"],
    columns="variable",
    values="value"
).reset_index()

merged_data.columns.name = None

# Keep days with both measurements
paired_data = merged_data.dropna(
    subset=["streamflow", "water_temperature"]
).copy()

# Sort chronologically
paired_data = paired_data.sort_values("date")

print("\n--- MERGED DATA RESULTS ---")

print("Total station-days:", len(merged_data))
print("Days with both measurements:", len(paired_data))

print("\nFirst five paired observations:")
print(paired_data.head())

# Classify streamflow conditions for each station

analysis_data = paired_data.copy()

# Calculate each station's streamflow thresholds
analysis_data["low_threshold"] = (
    analysis_data.groupby("site")["streamflow"]
    .transform(lambda x: x.quantile(0.25))
)

analysis_data["high_threshold"] = (
    analysis_data.groupby("site")["streamflow"]
    .transform(lambda x: x.quantile(0.75))
)

# Assign each day a flow category
def classify_flow(row):
    if row["streamflow"] >= row["high_threshold"]:
        return "High Flow"
    elif row["streamflow"] <= row["low_threshold"]:
        return "Low Flow"
    else:
        return "Normal Flow"

analysis_data["flow_category"] = analysis_data.apply(
    classify_flow, axis=1
)

print("Flow category counts:")
print(analysis_data["flow_category"].value_counts())

print("\nAverage water temperature by flow category:")
print(
    analysis_data.groupby("flow_category")[
        "water_temperature"
    ].agg(["count", "mean", "std"])
)



--- MERGED DATA RESULTS ---
Total station-days: 365
Days with both measurements: 335

First five paired observations:
            site area_type       date  streamflow  water_temperature
0  USGS-01435000   Unknown 2025-01-01       731.0                4.0
1  USGS-01435000   Unknown 2025-01-02       491.0                2.5
2  USGS-01435000   Unknown 2025-01-03       358.0                1.5
3  USGS-01435000   Unknown 2025-01-04       286.0                0.7
4  USGS-01435000   Unknown 2025-01-05       239.0                0.6
Flow category counts:
flow_category
Normal Flow    166
Low Flow        85
High Flow       84
Name: count, dtype: int64

Average water temperature by flow category:
               count      mean       std
flow_category                           
High Flow         84  7.678571  4.491842
Low Flow          85  9.169412  7.960049
Normal Flow      166  8.259639  6.485985


## PART 5: CHECK DATA QUALITY

In [7]:

print("\n--- DATA QUALITY CHECK ---")

print("\nMissing values:")
print(merged_data.isna().sum())

print("\nSummary statistics:")
print(
    paired_data[
        ["streamflow", "water_temperature"]
    ].describe()
)

# Confirm there is enough data for exploratory analysis
if len(paired_data) >= 30:
    print("\nSUCCESS: Sufficient paired observations for initial analysis.")
else:
    print("\nWARNING: Fewer than 30 paired observations.")

if AREA_TYPE == "Unknown":
    print(
        "NOTE: Site classification still needs verification. "
        "Urban/rural comparison is not ready."
    )


--- DATA QUALITY CHECK ---

Missing values:
site                  0
area_type             0
date                  0
streamflow            0
water_temperature    30
dtype: int64

Summary statistics:
        streamflow  water_temperature
count   335.000000         335.000000
mean    174.288060           8.344776
std     196.490375           6.482267
min      44.600000           0.100000
25%      77.400000           1.350000
50%     123.000000           7.600000
75%     183.500000          14.650000
max    1680.000000          20.100000

SUCCESS: Sufficient paired observations for initial analysis.
NOTE: Site classification still needs verification. Urban/rural comparison is not ready.


## PART 6: SAVE CLEANED DATA

In [8]:

merged_data.to_csv(
    DATA_DIR / "all_station_days.csv",
    index=False
)

paired_data.to_csv(
    DATA_DIR / "paired_streamflow_water_quality.csv",
    index=False
)

print("\nSaved cleaned datasets to:", DATA_DIR.resolve())

print("\nData collection and cleaning completed.")



Saved cleaned datasets to: /Users/laylarappleye/DS3000/DS-3000-Group-Project/data

Data collection and cleaning completed.


## PART 7: INVESTIGATE ADDITIONAL WATER QUALITY DATA

In [11]:
metadata_url = (
    "https://api.waterdata.usgs.gov/ogcapi/v1/"
    "collections/time-series-metadata/items"
)

water_quality_parameters = {
    "specific_conductance": "00095",
    "turbidity": "63680",
    "dissolved_oxygen": "00300"
}

for variable, parameter_code in water_quality_parameters.items():

    params = {
        "monitoring_location_id": SITE,
        "parameter_code": parameter_code,
        "statistic_id": "00003",
        "limit": 500,
        "f": "json"
    }

    response = requests.get(
        metadata_url,
        params=params,
        timeout=40
    )

    response.raise_for_status()

    features = response.json().get("features", [])

    print(f"\n--- {variable.upper()} ---")
    print("Daily mean time series found:", len(features))

    if features:
        metadata = pd.DataFrame([
            feature["properties"]
            for feature in features
        ])

        columns = [
            "id",
            "primary",
            "sublocation_identifier",
            "web_description"
        ]

        available = [
            col for col in columns
            if col in metadata.columns
        ]

        print(metadata[available].to_string(index=False))

    else:
        print("No daily mean time series found for this parameter.")



--- SPECIFIC_CONDUCTANCE ---
Daily mean time series found: 11
                              id primary sublocation_identifier web_description
02b8b70c77844d1f98f47103555e4733     NaN        Piezometer 1 WQ     Piezometer1
0384db582c7849efad35af6d03c30708 Primary                    NaN     Piezometer1
0747b9b1c9b54662a3770741c65aebfe     NaN        Piezometer 4 WQ     Piezometer4
4558b0d91ca3414ca0d0678c6b2a10c0 Primary                    NaN     Piezometer4
6cdf653292ac4729ab0576cd411f4a04 Primary                    NaN     Piezometer2
7caa9d4c54194d9aa549cacaceabf934 Primary                    NaN     Piezometer5
8e247bf4a8e1416f8a10a754d9dcbf14     NaN        Piezometer 5 WQ     Piezometer5
b990580737584dfd9bd639fe449c14f0 Primary             Channel WQ             NaN
c048a66df2c94c0c878f71ebb42e181a Primary                    NaN     Piezometer3
e56ce9914e324beab8631735b0740fa6     NaN        Piezometer 3 WQ     Piezometer3
fd79cd0946164e4ea85bd133f643f0e0     NaN                 